# 17 — Submission P7c: promedio 50/50 de CatBoost P6b + HistGB P7a

Ronda P7 (`laboratorio.ipynb`): el promedio simple de las probabilidades de dos modelos que codifican las categóricas de forma distinta mejoró a CatBoost solo:
- semilla de CV 42: **0,9376**, 5/5 folds, +0,0011;
- semilla de CV 7: 0,9373, 5/5 folds, +0,0013.

Los dos modelos:
- **CatBoost P6b**: `armar_pipeline_final()` de `features.py` (categóricas crudas + modelo de texto + numéricas).
- **HistGB P7a**: la Base HistGB, con `TargetEncoder(target_type="binary")` sobre `seller_id`, `category_id` y las 19 combinaciones de `agregar_combos`, más `VendedorEncoder(incluir_target=False)`. Se arma acá, igual que en el laboratorio; si se elige como final, pasa a `features.py`.

Promedio **50/50** (sin parámetros elegidos mirando datos). No toca `train.ipynb` ni `predict.ipynb`. **No sube nada.**

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import warnings
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, TargetEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier

from features import (
    aplanar_anidadas, agregar_features_contenido, agregar_categoricas_crudas, agregar_combos, agregar_logs,
    CapeoPorMoneda, VendedorEncoder, PrediccionOOF, armar_modelo_texto, armar_pipeline_final,
    COLUMNAS_NUMERICAS, COLUMNAS_COMBOS, INDICES_CAT,
)

SEED = 42

## 1. Datos (mismas funciones determinísticas en train y test)

In [2]:
def preparar(df):
    df = aplanar_anidadas(df)
    df = agregar_features_contenido(df)
    df = agregar_categoricas_crudas(df)
    return agregar_combos(df)


train = pd.read_json("../data/train_data.jsonlines", lines=True)
test = pd.read_json("../data/test_data.jsonlines", lines=True)
y = (train["condition"] == "new").astype(int)
train, test = preparar(train), preparar(test)

## 2. Los dos modelos, entrenados con todo el train

In [3]:
pipeline_histgb_p7a = Pipeline([
    ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
    ("logs", FunctionTransformer(agregar_logs)),
    ("preprocesador", ColumnTransformer([
        ("onehot", OneHotEncoder(handle_unknown="ignore"), ["listing_type_id", "buying_mode", "shipping_mode", "seller_state"]),
        ("num", "passthrough", COLUMNAS_NUMERICAS),
        ("tfidf", TfidfVectorizer(max_features=500), "title"),
        ("texto", PrediccionOOF(armar_modelo_texto(), random_state=SEED), ["title", "warranty_texto"]),
        ("te_combos", TargetEncoder(target_type="binary", random_state=SEED), ["seller_id", "category_id"] + COLUMNAS_COMBOS),
        ("vendedor_simple", VendedorEncoder(incluir_target=False), ["seller_id", "log_price"]),
    ], sparse_threshold=0)),
    ("modelo", HistGradientBoostingClassifier(learning_rate=0.1, max_depth=None, max_iter=200, random_state=SEED)),
])
pipeline_catboost = armar_pipeline_final()

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    pipeline_catboost.fit(train, y, modelo__cat_features=INDICES_CAT)
    pipeline_histgb_p7a.fit(train, y)

## 3. Promedio 50/50 y CSV

In [4]:
prob_catboost = pipeline_catboost.predict_proba(test)[:, 1]
prob_histgb = pipeline_histgb_p7a.predict_proba(test)[:, 1]
prob_promedio = 0.5 * prob_catboost + 0.5 * prob_histgb

submission = pd.DataFrame({
    "ID": np.arange(1, len(test) + 1),
    "condition": np.where(prob_promedio >= 0.5, "new", "used"),
})
Path("../submission").mkdir(exist_ok=True)
archivo = "../submission/submission_catboost_histgb.csv"
submission.to_csv(archivo, index=False)
print("Guardado:", archivo)

Guardado: ../submission/submission_catboost_histgb.csv


## 4. Chequeos

Formato, y comparación con `submission_catboost.csv` (CatBoost solo, primera en el público). La CV mejoró ~0,001, así que deberían cambiar del orden de cientos de filas. Además, la parte CatBoost sola tiene que reproducir exactamente `submission_catboost.csv`.

In [5]:
leida = pd.read_csv(archivo)
assert leida.shape == (30000, 2)
assert list(leida.columns) == ["ID", "condition"]
assert (leida["ID"].to_numpy() == np.arange(1, 30001)).all()
assert leida["condition"].isin(["new", "used"]).all()
print("Formato OK")

catboost_solo = pd.read_csv("../submission/submission_catboost.csv")
print("CatBoost de esta notebook == submission_catboost.csv:",
      (np.where(prob_catboost >= 0.5, "new", "used") == catboost_solo["condition"]).all())
cambian = (leida["condition"] != catboost_solo["condition"]).sum()
print(f"Predicciones distintas vs submission_catboost.csv: {cambian} ({cambian / len(leida):.1%})")
print("% new predicho:", round((leida["condition"] == "new").mean(), 4))

Formato OK
CatBoost de esta notebook == submission_catboost.csv: True
Predicciones distintas vs submission_catboost.csv: 266 (0.9%)
% new predicho: 0.5292
